# Data QA + DBSCAN Sizing

**Purpose:** the data-engineering checklist in `paper/TASKS.md` Section A — sort/clean GPS
pings with per-ping motion features (dt/dist/speed, implausible-jump flagging), QA the
mine-drawn zone polygons against actual GPS stop density, and use real zone sizes to pick a
DBSCAN `eps`/`min_samples` range for the stop-detection pipeline (Section C).

**What the next cell does:**

- `sys.path.append("../..")` — adds the repo root to `sys.path` so `gps_lib` (which lives two directories above `notebooks/analysis/`) can be imported.
- `from gps_lib import io_utils, classify, preprocess, zones` — imports the four helper modules used throughout this notebook:
  - `io_utils` — CSV read/write helpers for the tracker list, zone list, zone detail, and monthly GPS ping files.
  - `classify` — regex-based label classification (technic type/material, zone material/load type).
  - `preprocess` — GPS ping cleaning and per-ping motion-feature computation.
  - `zones` — zone polygon construction and point-in-zone lookups.

In [2]:
import sys
sys.path.append("../..")

from gps_lib import io_utils, classify, preprocess, zones

## 1. Load + clean GPS pings

**What this cell does, line by line:**

1. `io_utils.load_tracker_list()` — reads `tracker_list.csv` (the equipment registry: one row per tracker, with `id`, `label`, etc.).
2. `classify.classify_technic_material_type(...)` — adds a `technic_m_type` column, defaulting to `"other"` and tagging rows whose `label` matches `"BN"` (case-insensitive) as `"bn"`.
3. `io_utils.load_gps_data()` — globs `gps_data_<year>-<month>.csv` files in `DATA_DIR`, sorts them chronologically, and `pd.concat`s them into one DataFrame of raw GPS pings across all months.
4. `preprocess.attach_technic_info(...)` — inner-joins pings to tracker metadata on `tracker_id == id`. Pings whose `tracker_id` has no matching tracker row are dropped here.
5. `preprocess.clean_gps_points(..., round_n=4)` — the cleaning pipeline:
   - drops exact duplicate rows;
   - parses `get_time` with `pd.to_datetime` (bad values → `NaT`);
   - sorts by `tracker_id, get_time`;
   - derives `date` and `hour` columns;
   - rounds `lat`/`lng` to 4 decimals (~11m precision) to blur GPS jitter;
   - drops duplicates again on `(lat, lng, date, hour, tracker_id)`, collapsing near-duplicate pings (same tracker, same hour, within ~11m) down to one row.
6. `df_cleaned.shape` — sanity-check the row/column count that survived the pipeline.

Net effect: load raw multi-month GPS pings → tag trackers by material type → attach tracker metadata (dropping unmatched trackers) → parse/sort timestamps and collapse duplicate/near-duplicate pings to one row per tracker per ~11m location per hour.

In [3]:
tracker_list_df = io_utils.load_tracker_list()
tracker_list_df = classify.classify_technic_material_type(tracker_list_df)

track_points_df = io_utils.load_gps_data()
merged_df = preprocess.attach_technic_info(track_points_df, tracker_list_df)
df_cleaned = preprocess.clean_gps_points(merged_df, round_n=4)
df_cleaned.shape

/Users/uranbileg/Documents/GPS_project/notebooks/analysis/../../gps_lib/io_utils.py:46: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/Users/uranbileg/Documents/GPS_project/notebooks/analysis/../../gps_lib/io_utils.py:46: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/Users/uranbileg/Documents/GPS_project/notebooks/analysis/../../gps_lib/io_utils.py:46: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/Users/uranbileg/Documents/GPS_project/notebooks/analysis/../../gps_lib/io_utils.py:46: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.concat((pd.read_csv

(24081762, 30)

## 2. Per-ping motion features (dt / dist / speed) + implausible-jump flags

`add_motion_features` sorts by `tracker_id, get_time`, then computes seconds since the
previous ping, haversine distance from the previous ping, implied speed, and flags pings
implying speed above `max_speed_kmh` as likely GPS glitches.

**What this cell does, line by line:**

1. `preprocess.add_motion_features(df_cleaned, max_speed_kmh=120.0)` — sorts by `tracker_id, get_time`, then for each ping (per tracker, via `groupby("tracker_id").shift()`) computes:
   - `dt` — seconds elapsed since that tracker's previous ping.
   - `dist` — haversine distance (meters) from the previous ping's lat/lng to this one's.
   - `speed_kmh` — `dist / dt * 3.6`, the implied speed since the previous ping.
   - `implausible_jump` — `True` when `speed_kmh > max_speed_kmh` (120 km/h here), flagging likely GPS glitches. A tracker's very first ping has `dt`/`dist`/`speed_kmh` = `NaN` and `implausible_jump = False` (nothing to compare against).
2. `n_flagged = motion_df["implausible_jump"].sum()` — counts how many pings were flagged.
3. `print(...)` — reports the flagged count and its percentage of total pings.
4. `motion_df["speed_kmh"].describe()` — prints summary statistics (count, mean, std, min, quartiles, max) of implied speed, useful for judging whether the 120 km/h cutoff is reasonable for this fleet.

In [4]:
motion_df = preprocess.add_motion_features(df_cleaned, max_speed_kmh=120.0)

n_flagged = motion_df["implausible_jump"].sum()
print(f"Implausible-speed pings: {n_flagged:,} / {len(motion_df):,} "
      f"({n_flagged / len(motion_df):.3%})")
motion_df["speed_kmh"].describe()

Implausible-speed pings: 9,643 / 24,081,762 (0.040%)


count    2.408166e+07
mean              inf
std               NaN
min      0.000000e+00
25%      1.235312e+01
50%      2.469983e+01
75%      4.650824e+01
max               inf
Name: speed_kmh, dtype: float64

**What this cell does:**

Sorts `motion_df` by `speed_kmh` descending and shows the top 20 rows (`tracker_id`, `get_time`, `dt`, `dist`, `speed_kmh`, `implausible_jump`) — i.e. the pings with the most extreme implied speeds. This is a manual eyeball check on whether the `max_speed_kmh=120` threshold is catching genuine GPS glitches (huge `dist` over a tiny `dt`) rather than legitimate fast movement, before deciding whether to drop or just flag these pings downstream.

In [5]:
# Worst offenders — useful to eyeball before deciding whether to drop or just flag them
motion_df.sort_values("speed_kmh", ascending=False)[
    ["tracker_id", "get_time", "dt", "dist", "speed_kmh", "implausible_jump"]
].head(20)

,tracker_id,get_time,dt,dist,speed_kmh,implausible_jump
16126133,44403,2025-08-07 10:32:57,0.0,13.681451,inf,True
23335112,46078,2025-11-17 02:36:34,0.0,8.044724,inf,True
16410617,45294,2025-10-17 12:22:35,0.0,23.652021,inf,True
9320564,35364,2025-09-08 09:32:09,0.0,11.119493,inf,True
9350003,35364,2025-09-11 22:10:48,0.0,13.723973,inf,True
9599909,35364,2025-10-11 09:21:18,0.0,8.043612,inf,True
23334784,46078,2025-11-16 23:51:39,0.0,13.724712,inf,True
23334922,46078,2025-11-17 01:15:40,0.0,8.053481,inf,True
23335032,46078,2025-11-17 02:17:43,0.0,19.553156,inf,True
23335061,46078,2025-11-17 02:27:37,0.0,8.044965,inf,True


## 3. Zone polygon QA

Build the zone geometry, then check whether trucks actually stop inside each mine-drawn
zone polygon. Zones with ~0 stopped pings despite being labeled load/dump/fuel/repair are
candidates for stale or mis-drawn geometry.

**What this cell does, line by line:**

1. `io_utils.load_zone_list()` — reads `zone_list.csv`, the mine-drawn zone metadata table (`id`, `label`, etc.).
2. `classify.classify_zones(zone_list_df, drop_other=True)` — applies `classify_zone_material_type` (regex-tags each zone's `label` as `reject` / `bn` / `middling` / `other`), then `classify_zone_load_type` (tags each zone `load` / `unload`), and with `drop_other=True` discards any zone whose material type came out `"other"`.
3. `io_utils.load_zone_detail()` — reads `zone_detail_all_df.csv`, the raw polygon vertex list for every zone (`zone_id`, `lat`, `lng`, ...).
4. `zones.build_zone_geodataframe(zone_detail_all_df, zone_list_df)` — groups vertices by `zone_id`, builds a shapely `Polygon` for each zone with more than 2 points, wraps them all into a `geopandas.GeoDataFrame` (CRS `EPSG:4326`), and left-merges in `id`/`label`/`zone_material_type`/`zone_load_type` from `zone_list_df`.
5. `zones_gdf.shape` — sanity-check on the resulting zone-geometry table's row/column count.

In [6]:
zone_list_df = io_utils.load_zone_list()
zone_list_df = classify.classify_zones(zone_list_df, drop_other=True)
zone_detail_all_df = io_utils.load_zone_detail()

zones_gdf = zones.build_zone_geodataframe(zone_detail_all_df, zone_list_df)
zones_gdf.shape

(11, 6)

**What this cell does, line by line:**

1. `zones.zone_ping_density(motion_df, zones_gdf, speed_col="speed", speed_threshold=2.0)` — filters `motion_df` down to "stopped" pings (`speed_col < 2.0`, if that column exists), spatial-joins them against `zones_gdf` (`assign_zone_hit`, using a `geopandas.sjoin` with `predicate="within"` against the polygons' spatial index — needed to stay fast at fleet-scale ping counts), counts stopped pings per `zone_id`, and merges that count onto every zone in `zones_gdf` (zones with no hits get `0`). The result is sorted ascending by `n_stopped_pings`.
2. `print(...)` — reports how many zones have zero stopped pings — the QA signal: a zone labeled load/dump/fuel/repair with 0 stopped pings is a candidate for stale or mis-drawn polygon geometry.
3. `density_df.head(15)` — shows the 15 zones with the fewest stopped pings (most suspicious first), since the DataFrame is already sorted ascending.

In [7]:
density_df = zones.zone_ping_density(motion_df, zones_gdf, speed_col="speed", speed_threshold=2.0)
print("Zones with zero stopped pings (QA flag):", (density_df["n_stopped_pings"] == 0).sum())
density_df.head(15)

Zones with zero stopped pings (QA flag): 1


,zone_id,label,zone_material_type,zone_load_type,n_stopped_pings
0,27838,Баруун наран ачилтын бүс /Баруун/,bn,load,0
1,81702,Hard sp5,middling,unload,145
2,26140,SP4,middling,unload,1169
3,28368,Sp5,middling,unload,12667
4,82093,SP7 шинэ үүссэн,middling,unload,15463
5,25685,Баруун наран овоолго,bn,unload,27278
6,25385,Middling ачилтын бүс,middling,load,33029
7,28420,SP7,middling,unload,34472
8,25559,Баруун наран ачилтын бүс /Зүүн/,bn,load,53463
9,25334,Reject овоолго,reject,unload,99702


## 4. Zone size → DBSCAN eps/min_samples recommendation

Compute each zone's bounding-box diagonal (meters). `eps` should be small relative to the
*smallest* real zones (so adjacent zones don't merge into one cluster), and `min_samples`
should reflect the ping rate (~10-30s interval) over a plausible minimum dwell time.

**What this cell does:**

1. `zones.zone_diameter_stats(zones_gdf)` — for each zone polygon, takes its bounding box (`minx, miny, maxx, maxy`) and computes the haversine distance between opposite corners as `diameter_m` (an approximation of the zone's largest extent), returning a DataFrame sorted ascending by `diameter_m`.
2. `diam_df["diameter_m"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])` — prints summary statistics (count, mean, std, min, the requested percentiles, max) of zone diameters in meters, to see the spread of real zone sizes across the mine.

In [8]:
diam_df = zones.zone_diameter_stats(zones_gdf)
diam_df["diameter_m"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

count      11.000000
mean      792.083275
std       814.726944
min       284.868290
10%       292.009468
25%       306.416949
50%       428.876963
75%       896.767632
90%      1455.745051
max      2969.367299
Name: diameter_m, dtype: float64

**What this cell does:**

1. `smallest = diam_df.head(10)` — since `diam_df` is already sorted ascending by `diameter_m`, this grabs the 10 smallest zones. These impose the tightest upper bound on a usable DBSCAN `eps`: `eps` needs to stay well below the smallest real zone's size, or a stop-cluster could spill across into a neighboring zone.
2. `print(...)` — labels the output.
3. `smallest[["label", "zone_material_type", "diameter_m"]]` — displays each of those 10 zones' label, material type, and diameter for manual inspection.

In [9]:
smallest = diam_df.head(10)
print("Smallest zones (tightest constraint on eps):")
smallest[["label", "zone_material_type", "diameter_m"]]

Smallest zones (tightest constraint on eps):


,label,zone_material_type,diameter_m
0,SP4,middling,284.868290
1,Reject ачилтын бүс,reject,292.009468
2,Sp5,middling,298.911379
3,Middling ачилтын бүс,middling,313.922519
4,Hard sp5,middling,326.190795
5,Баруун наран овоолго,bn,428.876963
6,SP7 шинэ үүссэн,middling,549.488993
7,SP7,middling,742.097133
8,Баруун наран ачилтын бүс /Зүүн/,bn,1051.438131
9,Баруун наран ачилтын бүс /Баруун/,bn,1455.745051


**Recommendation:** set DBSCAN `eps` to roughly half the smallest real zone diameter
(`diam_df["diameter_m"].quantile(0.1) / 2`, printed above) so clusters stay within a single
zone's footprint, and set `min_samples` to the number of pings expected during a minimum
plausible dwell (e.g. ~60s dwell / ~20s ping interval ≈ 3). Tune both against the
parameter-sensitivity sweep in `paper/TASKS.md` Section C once the DBSCAN pipeline exists —
treat these as a starting search range, not a final answer.